<a href="https://colab.research.google.com/github/cleomarbrdias/script/blob/main/Captura%20Experiencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# OCR do PDF (captura de tela) -> Excel (Colab)
# Extrai: ID, Título, Núcleo, Situação, Avaliadores, Notas, Média
# ============================================

# 1) Instalar dependências do OCR e utilitários
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr poppler-utils > /dev/null
!pip -q install pytesseract pdf2image pandas openpyxl lxml

import re, os, io, pandas as pd, pytesseract
from pdf2image import convert_from_path
from google.colab import files
from PIL import Image

# 2) Upload do PDF (use sua captura em PDF)
print("Faça upload do seu PDF (a captura da tela):")
uploaded = files.upload()  # selecione o arquivo .pdf
assert uploaded, "Nenhum PDF enviado."
pdf_name = list(uploaded.keys())[0]

# 3) Converter páginas do PDF para imagens (300 DPI para melhor OCR)
pages = convert_from_path(pdf_name, dpi=300)
print(f"Páginas convertidas: {len(pages)}")

# 4) OCR de cada página (modo linha de leitura boa para blocos)
#    Usamos a saída em texto (não TSV) e depois regex para encontrar os campos.
full_text = ""
for i, img in enumerate(pages, start=1):
    # ligeiro aumento de contraste/tons de cinza melhora OCR em screenshots
    gray = img.convert("L")
    # psm 6 = Assume um bloco uniforme de texto; oem 3 = LSTM default
    txt = pytesseract.image_to_string(
        gray,
        lang="por",
        config="--oem 3 --psm 6"
    )
    full_text += "\n\n" + txt

# 5) Normalização básica do texto OCR
def norm(s: str) -> str:
    s = re.sub(r'[ \t]+', ' ', s)
    s = s.replace('\r', '')
    # colapsa quebras múltiplas
    s = re.sub(r'\n{3,}', '\n\n', s)
    return s.strip()

full_text = norm(full_text)

# 6) Separar cada item pelo padrão do ID (linhas com 4–6 dígitos em destaque)
#    Estratégia: quebrar sempre que encontramos uma linha "pura" com um número 4–6 dígitos.
#    Depois, para cada bloco, extrair campos por regex.
#    Observação: como é OCR, deixamos as regex tolerantes a acentuação/variações.
blocks = []
# Inserimos um marcador antes de linhas que pareçam "ID" puro para facilitar split
marker_text = re.sub(r'(?m)^\s*(\d{4,6})\s*$', r'\n@@ITEM_ID_\1@@\n', full_text)
parts = marker_text.split('@@ITEM_ID_')

for part in parts:
    part = part.strip()
    if not part or '@@' not in part:
        continue
    id_str, body = part.split('@@', 1)
    item_id = re.sub(r'\D+', '', id_str)  # só dígitos
    block_text = norm(body)
    if item_id:
        blocks.append((item_id, block_text))

# 7) Funções de extração por bloco
def extrai_campos(item_id: str, text: str) -> dict:
    # Média
    m_media = re.search(r"M[eé]dia\s*:\s*([0-9]+)", text, flags=re.I)
    media = m_media.group(1) if m_media else ""

    # Notas (pega todas)
    notas = re.findall(r"Nota\s*:\s*([0-9]+)", text, flags=re.I)
    notas_join = "; ".join(notas)

    # Núcleo: de "Núcleo" até encontrar um rótulo típico ou fim
    m_nucleo = re.search(
        r"(N[úu]cleo[\s\S]{0,200})(?:\n\s*Homolog|Homolog|Avaliador|Nota|M[eé]dia|$)",
        text, flags=re.I
    )
    nucleo = norm(m_nucleo.group(1)) if m_nucleo else ""

    # Situação (Homologação)
    m_sit = re.search(
        r"(Homologada|N[aã]o\s*Homologada|Pendente[\s\S]{0,30}?homologa[cç][aã]o)",
        text, flags=re.I
    )
    situacao = norm(m_sit.group(1)) if m_sit else ""

    # Avaliadores (todas as ocorrências)
    avaliadores = [re.sub(r"^Avaliador\s*:\s*", "", a, flags=re.I).strip()
                   for a in re.findall(r"Avaliador\s*:\s*([^\n]+)", text, flags=re.I)]
    avaliadores_join = "; ".join(avaliadores)

    # Título: heurística — tudo entre o início do bloco e o começo do "Núcleo"
    titulo = ""
    low = text.lower()
    ix_nucleo = low.find("núcleo")
    if ix_nucleo > 0:
        titulo = text[:ix_nucleo]
    else:
        # se não achar Núcleo, pega primeiras 3 linhas significativas
        linhas = [l.strip() for l in text.split("\n") if l.strip()]
        titulo = "\n".join(linhas[:3])

    # limpa rótulos residuais
    titulo = re.sub(r"(^|\n)\s*(Avaliador|Nota|M[eé]dia)\s*:.*", "", titulo, flags=re.I)
    titulo = re.sub(r"(^|\n)\s*(Homologada|N[aã]o\s*Homologada|Pendente[\s\S]{0,30}?homologa[cç][aã]o).*", "", titulo, flags=re.I)
    titulo = norm(titulo)

    return {
        "ID": item_id,
        "Título": titulo,
        "Núcleo": nucleo,
        "Situação": situacao,
        "Avaliadores": avaliadores_join,
        "Notas": notas_join,
        "Média": media
    }

# 8) Extrair todos os blocos
rows = []
for item_id, body in blocks:
    row = extrai_campos(item_id, body)
    # descarta falsos positivos (sem título e sem média e sem avaliador)
    if row["Título"] or row["Média"] or row["Avaliadores"]:
        rows.append(row)

df = pd.DataFrame(rows).drop_duplicates(subset=["ID", "Título"]).reset_index(drop=True)

# 9) Salvar Excel e exibir prévia
out_path = "/content/lista_criaj_ocr.xlsx"
df.to_excel(out_path, index=False)
print(f"✅ Registros extraídos: {len(df)}")
print(f"📄 Planilha salva em: {out_path}")

df.head(12)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Faça upload do seu PDF (a captura da tela):


Saving FireShot Capture 022 - lista-criaj – Eventos - [apsredes.org].pdf to FireShot Capture 022 - lista-criaj – Eventos - [apsredes.org].pdf


DecompressionBombError: Image size (403746000 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack.

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/1iLT7PkYgwTxapl1ZXZrjx3zvaRFNVC8ew4Bq7uVwuq8/edit#gid=0
